In [1]:
# dotnet install / check
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

home = Path.home()
dotnet_cmd = shutil.which("dotnet")
if dotnet_cmd is None:
    bundled = home / ".dotnet" / ("dotnet.exe" if os.name == "nt" else "dotnet")
    if bundled.exists():
        dotnet_cmd = str(bundled)

if dotnet_cmd is None:
    install_script = home / "dotnet-install.sh"
    urllib.request.urlretrieve("https://dot.net/v1/dotnet-install.sh", install_script)
    subprocess.run(["bash", str(install_script), "--channel", "10.0", "--install-dir", str(home / ".dotnet")], check=True)
    dotnet_cmd = str(home / ".dotnet" / ("dotnet.exe" if os.name == "nt" else "dotnet"))

info = subprocess.run([dotnet_cmd, "--info"], capture_output=True, text=True, check=True)
print(info.stdout)

.NET SDK:
 Version:           10.0.201
 Commit:            4d3023de60
 Workload version:  10.0.200-manifests.0793c108
 MSBuild version:   18.3.0-release-26153-122+4d3023de6

런타임 환경:
 OS Name:     ubuntu
 OS Version:  24.04
 OS Platform: Linux
 RID:         linux-x64
 Base Path:   /home/20212052/.dotnet/sdk/10.0.201/

설치된 .NET 워크로드:
표시할 설치된 워크로드가 없습니다.
새 매니페스트를 설치할 때 use workload sets 사용하도록 구성됩니다.
워크로드 집합이 설치되어 있지 않습니다. "dotnet workload restore"를 실행하여 워크로드 집합을 설치하세요.

Host:
  Version:      10.0.5
  Architecture: x64
  Commit:       a612c2a105

.NET SDKs installed:
  10.0.201 [/home/20212052/.dotnet/sdk]

.NET runtimes installed:
  Microsoft.AspNetCore.App 10.0.5 [/home/20212052/.dotnet/shared/Microsoft.AspNetCore.App]
  Microsoft.NETCore.App 10.0.5 [/home/20212052/.dotnet/shared/Microsoft.NETCore.App]

Other architectures found:
  None

Environment variables:
  Not set

global.json file:
  Not found

Learn more:
  https://aka.ms/dotnet/info

Download .NET:
  https://aka.ms/dotnet/downlo

In [2]:
#노트북 커널 기준 패키지 확인 / 설치
import importlib.util
import subprocess
import sys
from pathlib import Path

deps_dir = Path.home() / ".rl_ai_deps"
deps_dir.mkdir(parents=True, exist_ok=True)
if str(deps_dir) not in sys.path:
    sys.path.insert(0, str(deps_dir))

required = ["torch", "numpy", "pytest", "pythonnet", "clr_loader"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--target", str(deps_dir), *missing],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    if completed.returncode != 0:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(f"pip install failed with exit code {completed.returncode}")

import torch
import numpy
import setuptools

print(sys.executable)
print(torch.__version__)
print(numpy.__version__)
print(setuptools.__version__)
print(torch.cuda.is_available())


/opt/python/bin/python
2.11.0+cu130
2.3.5
82.0.0
True


In [3]:
# 압축 해제
import shutil
import tempfile
import zipfile
from pathlib import Path

home = Path.home()
zip_candidates = [Path.cwd() / "RL_AI.zip", home / "RL_AI.zip"]
zip_path = next((path for path in zip_candidates if path.exists()), None)

target_dir = home / "RL_AI"
if zip_path is None:
    print("RL_AI.zip not found, skipping unzip")
else:
    if target_dir.exists():
        shutil.rmtree(target_dir)
    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_path = Path(tmp_dir)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp_path)

        nested_root = tmp_path / "RL_AI"
        source_root = nested_root if nested_root.exists() else tmp_path
        target_dir.mkdir(parents=True, exist_ok=True)
        for item in source_root.iterdir():
            shutil.move(str(item), str(target_dir / item.name))
    print("RL_AI ready")


RL_AI ready


In [4]:
# C# build
import os
import shutil
import subprocess
from pathlib import Path

home = Path.home()
dotnet_cmd = shutil.which("dotnet")
if dotnet_cmd is None:
    bundled = home / ".dotnet" / ("dotnet.exe" if os.name == "nt" else "dotnet")
    if bundled.exists():
        dotnet_cmd = str(bundled)
if dotnet_cmd is None:
    raise FileNotFoundError("dotnet not found. Run the first cell again.")

project_root = home / "RL_AI" / "SeaEngine" / "csharp"
cli_csproj = project_root / "SeaEngineCli" / "SeaEngineCli.csproj"
engine_csproj = project_root / "SeaEngine" / "SeaEngine.csproj"

subprocess.run([dotnet_cmd, "build", str(cli_csproj), "-c", "Debug", "-v", "q"], check=True)
subprocess.run([dotnet_cmd, "build", str(engine_csproj), "-c", "Release", "-v", "q"], check=True)
print("SeaEngine build ok")



빌드했습니다.
    경고 0개
    오류 0개

경과 시간: 00:00:16.16

빌드했습니다.
    경고 0개
    오류 0개

경과 시간: 00:00:02.24
SeaEngine build ok


In [5]:
#평가 / 학습
import importlib
import os
import sys
from pathlib import Path
from shutil import which

home = Path.home()
if str(home) not in sys.path:
    sys.path.insert(0, str(home))

dotnet_cmd = which("dotnet")
if dotnet_cmd is None:
    fallback = home / ".dotnet" / ("dotnet.exe" if os.name == "nt" else "dotnet")
    if fallback.exists():
        dotnet_cmd = str(fallback)
if dotnet_cmd:
    os.environ["DOTNET_CMD"] = dotnet_cmd
    dotnet_root = str(Path(dotnet_cmd).resolve().parent)
    os.environ.setdefault("DOTNET_ROOT", dotnet_root)
    os.environ.setdefault("DOTNET_ROOT_X64", dotnet_root)

# zip을 다시 풀었거나 파일을 교체한 경우에도, 이전에 import된 RL_AI 모듈 캐시를 지웁니다.
for module_name in list(sys.modules):
    if module_name == "RL_AI" or module_name.startswith("RL_AI."):
        del sys.modules[module_name]
importlib.invalidate_caches()

from RL_AI.SeaEngine.experiment import run_checkpoint_training_experiment, run_train_eval_experiment
from RL_AI.SeaEngine import trainer as seaengine_trainer_module
from RL_AI.SeaEngine.bridge import pythonnet_session as pythonnet_session_module
print(f"trainer source: {seaengine_trainer_module.__file__}")
print(f"pythonnet source: {pythonnet_session_module.__file__}")


EVAL_MATCHES = 50
MAX_TURNS = 100
TRAIN_EPISODES = 10000
UPDATE_INTERVAL = 16

# 학습 전/후를 한 번에 저장하는 기본 실험
result = run_train_eval_experiment(
    eval_matches=EVAL_MATCHES,
    train_episodes=TRAIN_EPISODES,
    max_turns=MAX_TURNS,
    update_interval=UPDATE_INTERVAL,
    seed=7,
)

print("=== SeaEngine Train/Eval Experiment ===")
print(f"summary report: {result['report_path']}")
print(f"before_random: {result['before_random']['report_path']}")
print(f"before_greedy: {result['before_greedy']['report_path']}")
print(f"after_random: {result['after_random']['report_path']}")
print(f"after_greedy: {result['after_greedy']['report_path']}")
print(result['train'])

# checkpoint까지 보고 싶으면 아래를 따로 실행하세요.
# checkpoint = run_checkpoint_training_experiment(
#     eval_matches=100,
#     total_train_episodes=10000,
#     eval_interval=500,
#     max_turns=100,
#     update_interval=16,
#     seed=7,
# )
# print(checkpoint['summary_report_path'])


trainer source: /home/20212052/RL_AI/SeaEngine/trainer.py
pythonnet source: /home/20212052/RL_AI/SeaEngine/bridge/pythonnet_session.py
[*] Experiment start | eval_matches=100 | train_episodes=10000 | max_turns=100 | update_interval=16 | num_envs=4 | vector_backend=local | local_threads=1
[*] Evaluating before training vs random...



KeyboardInterrupt

